# 03 · Validation and conclusions

**PRT2030 · Dataset inspection · Study notebook**

Notebook 02 checked representations: could we interpret values as labels, numbers or dates? Now test their meaning.
We will establish which records are eligible for each check, apply reference rules, inspect coverage and relationships,
and investigate unusual sizes. We finish by documenting a year disagreement and deciding what analysis the data can support.

**A check can pass, flag a concern, or remain untested.** A missing prerequisite is not a pass. A review flag is not proof of an error.

**How to use this study resource:** Read the inspection question, follow the comments as you run the code,
and explain what the output tells you. Run from the top; later checks reuse named results from earlier cells.
Each notebook loads its own data and runs independently. Use your existing `wildfires.csv` beside the notebooks.

**One source row represents a reported wildfire record.** Matching names or rows do not establish that records
describe the same physical fire: this extract has no incident identifier. Fire sizes are in acres.
We inspect temporary results and evidence tables while keeping `raw` unchanged.

The worked findings refer to the supplied teaching extract (1,880,465 records and eight columns).
They are reference figures from the supplied notebooks, not results of a new run in this revision.
Code outputs have been cleared so your next top-to-bottom run produces a consistent set of results.
If your file differs, use your own computed results. The teaching target remains pandas 3.0.5.
The worked log wording describes the reference findings; revise the written judgements if your results differ.

Function and method names in the documentation lines are clickable links to their specific official reference pages.

**What does “rule” mean here?** It is an explicit condition we use to accept a value or request
review, with a reason tied to the field or analysis. For example, a reported year must be whole
and fall in this extract's 1992–2015 period; a 25% annual-change threshold merely requests context.
The first expectation follows the extract definition. The second is a classroom screening choice.
Examples below make both the condition and the population it can test visible.

**Reading the examples:** The small values introduced as examples explain the logic;
they are not new measurements from the wildfire file. Comments explain the purpose,
intermediate results and reasons for the choices. Apply the stated prerequisites
before interpreting a zero flag count as a pass.


## 1. Load the data

Load a fresh original table. We recreate the comparison values within this notebook so it can run independently.

**Documentation:** [pd.read_csv()](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [len()](https://docs.python.org/3.14/library/functions.html#len).


In [ ]:
# We need a trustworthy starting table before we can judge its values. Keep this import as
# raw so every finding can be traced back to the same original records.

# Pandas gives us labelled tables and columns. The short name pd lets us use that library
# consistently in all later cells.
import pandas as pd

# The CSV becomes a DataFrame: one row is one reported fire, and each column supplies a
# different fact about it. Loading it afresh lets this notebook run independently of the
# others.
raw = pd.read_csv("wildfires.csv")

# A missing-value check asks about records, so its whole-file denominator must be the row
# count. Counting all cells would make the affected percentage artificially small.
total_records = len(raw)
# Stop on an empty import: there is no population to inspect and division by the row count
# would be undefined. An empty file cannot demonstrate that its values passed a check.
if total_records == 0:
    raise ValueError("The dataset contains no records. Check the import before continuing.")

# Record the environment alongside the count so an unexpected result can be compared with
# another run. A mixed-type warning is a clue to inspect a column; it does not identify which
# values are wrong.
print("pandas version:", pd.__version__)
print("Reported wildfire records:", total_records)


## 2. Prepare values for meaningful checks

Year and size rules need finite numbers; date relationships need interpretable dates. Notebook 02 explained conversion
failures. Here we recreate those temporary values and explicitly identify which records can participate in later rules.

**Prerequisite examples:** `1998.5` is finite and can enter the whole-year check, where it is
flagged as fractional. Infinity and an unconvertible `unknown` cannot enter that finite-number
check. Likewise, a record with no usable containment date cannot enter the two-endpoint order
check. Missing a prerequisite means **untested for that check**, not accepted.


### 2.1 Prepare numerical values and eligibility flags

A missing, unconvertible or infinite value cannot support an ordinary year or size range check. Exclude it from that comparison while retaining it for the separate representation finding.

**Inspection question:** Which records contain finite year and size values?

**Documentation:** [pd.to_numeric()](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html) · [np.isfinite()](https://numpy.org/doc/stable/reference/generated/numpy.isfinite.html).


In [ ]:
# Meaning-based comparisons need actual finite numbers; a readable text value or infinity is
# not enough to enter an ordinary range check.

import numpy as np

# Convert into temporary Series while retaining the source-row labels. Coercion makes
# unavailable or unconvertible values missing, so later prerequisites can keep them out of
# numerical rules.
year_numbers = pd.to_numeric(raw["fire_year"], errors="coerce")
size_numbers = pd.to_numeric(raw["fire_size"], errors="coerce")

# isfinite returns True for an ordinary number such as 1998 or 1998.5 and False for missing
# values or infinity. It establishes eligibility for numerical inspection, not whether the
# number is a valid calendar year.
finite_year = np.isfinite(year_numbers)
finite_size = np.isfinite(size_numbers)
# Count the eligible population explicitly. We cannot report non-finite values as having
# passed a range comparison that did not meaningfully test them.
print("Finite years:", finite_year.sum())
print("Finite sizes:", finite_size.sum())


### 2.2 Prepare date values

We will compare date order and extract discovery years. Both operations require interpreted dates rather than raw strings.

**Inspection question:** How many discovery and containment dates are available for date-based rules?

**Documentation:** [pd.to_datetime()](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) · [Series.notna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.notna.html).

**Example of a date-based rule:** Containment should not be earlier than discovery for a
discovery-to-containment duration. Discovery on 10 July and containment on 12 July meet that
order condition; containment on 9 July is flagged; absent containment makes the comparison untested.
We prepare interpreted dates here and apply the order check in section 6.2.


In [ ]:
# We must interpret the date strings before extracting their years or comparing chronological
# order.

# The explicit layout tells Pandas how to interpret year, month, day and time. An impossible
# value becomes NaT under coercion; the source string remains in raw for investigation.
discovery_dates = pd.to_datetime(
    raw["discovery_date"], format="%Y-%m-%d %H:%M:%S", errors="coerce"
)
# Use the same conversion for both endpoints so a comparison concerns dates rather than the
# alphabetical order of their text.
contained_dates = pd.to_datetime(
    raw["contained_date"], format="%Y-%m-%d %H:%M:%S", errors="coerce"
)

# Count non-missing conversion results because only these can supply a year or endpoint.
# These counts combine availability and successful conversion; Notebook 02 separates absence
# from conversion failure.
print("Usable discovery dates:", discovery_dates.notna().sum())
print("Usable containment dates:", contained_dates.notna().sum())


## 3. Check categories against an independent expectation

Observed category lists describe the data; reference lists define the values permitted by this source edition.
Use the reference lists below rather than constructing an allowed list from the values we want to test.


### 3.1 Begin with the seven size classes

Class labels should be A through G. Missing classes are a separate completeness problem, so the membership rule checks only present labels.

**Inspection question:** Are any present size-class labels outside the allowed list?

**Documentation:** [Series.notna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.notna.html) · [Series.isin()](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html).

**Membership rule examples:**

| Reported class | Decision for this check | Reason |
| --- | --- | --- |
| `A` | Accepted | Present and in the documented A–G list. |
| `H` | Flagged | Present but outside that list. |
| Missing | Untested for membership | There is no present label to compare. |

An accepted class label still needs a separate size–class relationship check.


In [ ]:
# A category membership check asks whether each present class belongs to an independently
# documented list.

# Use A through G from the class definition. Building this list from observed values would
# accept an unexpected H merely because H appeared in the file.
allowed_classes = ["A", "B", "C", "D", "E", "F", "G"]

# A missing class cannot be tested as a present label. Record that prerequisite separately
# from membership.
class_present = raw["fire_class"].notna()
# Each present A-G label gets True; H gets False. isin tests complete values, so a padded " A
# " does not equal A under this exact membership check.
class_allowed = raw["fire_class"].isin(allowed_classes)

# Combine presence AND NOT allowed. A missing class stays outside this invalid-label result
# and remains untested for membership rather than becoming a pass.
invalid_class = class_present & ~class_allowed
print("Present classes checked:", class_present.sum())
print("Invalid class labels:", invalid_class.sum())
print("Missing, so untested for membership:", (~class_present).sum())
# Show original failing labels to reveal whether the cause might be spelling, padding or a
# genuinely undocumented category. Zero failures supports a pass only among the present
# labels actually checked.
display(raw.loc[invalid_class, ["fire_class"]].head(10))


### 3.2 Check the allowed jurisdictions and causes

Apply the same rule to the source edition's 50 states plus DC and Puerto Rico, and its 13 cause labels. Missing/Undefined is an allowed cause label, although it still limits cause analysis.

**Inspection question:** Do present state and cause labels belong to the documented categories?

**Documentation:** [str.split()](https://docs.python.org/3.14/library/stdtypes.html#str.split) · [Series.isin()](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html).

**Examples of the same membership rule:** `CA` belongs to the allowed jurisdictions; `ZZ` does
not. `Lightning` and `Missing/Undefined` both belong to the documented cause labels.
The latter is therefore accepted for **membership**, while C01 still records that its cause
is unknown. A missing code or cause is outside this present-label check.


In [ ]:
# Apply the same independent-reference logic to jurisdictions and causes, keeping a valid
# category distinct from an informative category.

# Adjacent text fragments form one string; splitting on whitespace makes a list of codes to
# compare with each source value. CA is in the reference; ZZ is not.
allowed_states = (
    "AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA ME MD MA "
    "MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD "
    "TN TX UT VT VA WA WV WI WY DC PR"
).split()
# Missing/Undefined belongs to the source cause list. It can therefore pass membership while
# still failing to identify a known cause for an analysis; C01 records that limitation
# separately.
allowed_causes = [
    "Lightning", "Equipment Use", "Smoking", "Campfire", "Debris Burning",
    "Railroad", "Arson", "Children", "Miscellaneous", "Fireworks",
    "Powerline", "Structure", "Missing/Undefined"
]

state_present = raw["state"].notna()
# Require a present code that is absent from the allowed list. Repeat exactly this logic for
# cause labels so absence and invalid membership remain separate findings.
invalid_state = state_present & ~raw["state"].isin(allowed_states)
cause_present = raw["fire_cause"].notna()
invalid_cause = cause_present & ~raw["fire_cause"].isin(allowed_causes)
# Report checked, flagged and untested counts together. These explain the scope of a
# zero-failure result and stop unavailable inputs from being mistaken for validated labels.
print("State: checked / invalid / untested:",
      state_present.sum(), invalid_state.sum(), (~state_present).sum())
print("Cause: checked / invalid / untested:",
      cause_present.sum(), invalid_cause.sum(), (~cause_present).sum())

# Review original values before proposing any mapping. A label outside this edition might
# indicate a new category, formatting difference or source problem.
display(raw.loc[invalid_state, ["state"]].head(10))
display(raw.loc[invalid_cause, ["fire_cause"]].head(10))


### 3.3 Separate code format, membership and absence

A two-letter uppercase code can have the right shape but still be invalid. Conversely, an allowed code need not occur in every extract. Inspect these separately.

**Inspection question:** Are codes correctly shaped, and which allowed categories never appear?

**Documentation:** [Series.str.fullmatch()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.fullmatch.html) · [Series.dropna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.dropna.html) · [set()](https://docs.python.org/3.14/library/stdtypes.html#set).


In [ ]:
# Separate three questions: is a code written in the expected shape, is it an allowed code,
# and does an allowed code appear in this extract?

# The pattern requires exactly two uppercase letters. CA and ZZ both fit; ca and CAA do not.
# Shape alone cannot establish allowed membership.
correct_state_shape = raw["state"].str.fullmatch(r"[A-Z]{2}", na=False)
# Exclude missing codes from shape failures using the presence condition. An absent value has
# no text layout to check.
bad_state_shape = state_present & ~correct_state_shape
print("Present codes with the wrong shape:", bad_state_shape.sum())

# Allowed minus observed finds permitted categories with no occurrence. Unlike invalid_state,
# this is a question about coverage, not a claim that a supplied value is wrong.
unobserved_states = set(allowed_states) - set(raw["state"].dropna())
print("Allowed states absent:", unobserved_states)
print("Allowed classes absent:", set(allowed_classes) - set(raw["fire_class"].dropna()))
# An allowed cause can legitimately have no records in a particular extract. Investigate the
# scope before treating its absence as missing incidents.
print("Allowed causes absent:", set(allowed_causes) - set(raw["fire_cause"].dropna()))


## 4. Check numeric meaning and ranges

Category labels have been compared with references. Now apply meaning-based rules to numeric values:
a reported year must be a whole year in this extract's period, and a fire size must be positive and finite.

**Two meaningful numeric rules:** For this extract, years must be finite whole values in
1992–2015, while sizes must be finite positive acres. Thus `1998.5` can be numerically readable
yet unsuitable as a reported year; `-2` can be numerically readable yet unsuitable as a positive
area. Representation and meaning answer different questions.


### 4.1 Validate years

Numeric conversion would accept 1998.5 or 2020, but neither meets the reported-year rule for this 1992–2015 extract.

**Inspection question:** Are any finite years fractional or outside the documented period?

**Documentation:** [Series.between()](https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html).

**Year rule examples:**

| Converted year | Whole-year result | Period result | Eligible as a valid reported year? |
| --- | --- | --- | --- |
| `1998` | Accepted | In range | Yes. |
| `1998.5` | Flagged | In range | No: fractional. |
| `2020` | Accepted | Flagged | No: outside this extract's period. |
| Missing or infinity | Untested | Untested | No: non-finite. |

The period includes both 1992 and 2015. A different extract needs its own documented period.


In [ ]:
# A valid reported year must meet several conditions at once: finite, whole, and within the
# teaching extract period.

# The remainder after division by 1 is zero for 1998 and 0.5 for 1998.5. A non-zero remainder
# flags a fractional value only when the numeric input is finite.
fractional_year = finite_year & (year_numbers % 1 != 0)

# 1992 and 2015 are included; 1991 and 2016 fall outside. This period belongs to this extract
# definition, not a universal rule for all wildfire datasets.
year_outside_range = finite_year & ~year_numbers.between(1992, 2015)

# Use AND so passing one condition cannot hide failure of another. 1998.5 is inside the
# period but fractional; 2020 is whole but outside. Neither becomes eligible as a valid
# reported year.
valid_year = finite_year & ~fractional_year & ~year_outside_range
print("Finite years checked:", finite_year.sum())
print("Fractional years:", fractional_year.sum())
print("Years outside 1992–2015:", year_outside_range.sum())
# Non-finite inputs did not enter these meaning checks. Keep their representation or
# missingness findings visible instead of describing them as good years.
print("Untested non-finite years:", (~finite_year).sum())


### 4.2 Validate sizes

A finite value can still be zero or negative. Establish the positive-size population before checking size classes and distributions.

**Inspection question:** Which finite sizes violate the positive-size rule?

**Documentation:** [Series.sum()](https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html) · [DataFrame.loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html).

**Positive-size rule examples:** A finite `0.1` acres meets the positive-size condition.
Finite `0` and `-2` acres are flagged. A missing or infinite size is untested for this finite-size
meaning check and is covered by the earlier availability/representation checks. Positive does
not automatically mean accurate.


In [ ]:
# A readable measurement must also satisfy the positive-size definition before entering class
# and distribution checks.

# A finite -2 or 0 acres is flagged by this rule; a finite 0.1 acres meets it. Missing and
# infinite values remain outside the checked finite population.
non_positive_size = finite_size & (size_numbers <= 0)
# Keep the acceptable population as its own mask because later calculations need those
# records. This is a temporary eligibility choice, not a deletion from raw.
valid_size = finite_size & (size_numbers > 0)
print("Finite sizes checked:", finite_size.sum())
print("Zero or negative sizes:", non_positive_size.sum())
print("Untested non-finite sizes:", (~finite_size).sum())
# Inspect the original zero or negative measurements before any treatment. A positive value
# passes this rule but might still be implausible, misreported or assigned the wrong class.
display(raw.loc[non_positive_size, ["fire_size"]].head(10))


## 5. Inspect coverage across years and jurisdictions

Usable individual values do not guarantee useful coverage of all groups. We will count records across the expected
years, examine large changes and look for empty jurisdiction–year combinations. Counts include unresolved repeated records.


### 5.1 Count records per expected year — V01

We need a valid reported year and an allowed jurisdiction for this coverage comparison. Include every expected year in the report, even when no eligible records occur in it.

**Inspection question:** Are any expected years absent from the eligible records?

**Documentation:** [range()](https://docs.python.org/3.14/library/stdtypes.html#range) · [Series.astype()](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html) · [Series.value_counts()](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) · [Series.reindex()](https://pandas.pydata.org/docs/reference/api/pandas.Series.reindex.html).


In [ ]:
# Count comparable records in every expected year so missing groups remain visible instead of
# disappearing from a frequency report.

# Require a valid reported year AND an allowed jurisdiction. This avoids comparing a year
# label or jurisdiction that has already failed its prerequisite rule.
coverage_eligible = valid_year & raw["state"].isin(allowed_states)

coverage_data = raw.loc[coverage_eligible, ["fire_year", "state"]].copy()

# Use the converted whole-year values in a separate table so 1998 and text "1998" group under
# the same numeric label. Conversion to integer is safe here because fractional, missing and
# infinite years were excluded.
coverage_data["fire_year"] = year_numbers.loc[coverage_eligible].astype("int64")

# range stops before 2016, so the list contains 1992 through 2015. These expectations come
# from the extract period, not whichever years happen to occur.
expected_years = list(range(1992, 2016))

# After counting, each item means a YEAR and its number of eligible source records. The
# result no longer has one item per fire.
year_counts = coverage_data["fire_year"].value_counts()

# Frequency counts alone omit years with no records and sort by frequency. Reindex restores
# calendar order and inserts zero for missing years; we need that order for the next-year
# comparison.
year_counts = year_counts.reindex(expected_years, fill_value=0)
print("Records checked:", coverage_eligible.sum())
print("Records excluded by prerequisites:", (~coverage_eligible).sum())
display(year_counts)

# A zero means no eligible records in this extract for that year. It does not establish that
# no physical fires happened; reporting coverage and the eligibility exclusions need
# investigation.
empty_years = year_counts.loc[year_counts == 0]
print("Years with no eligible records:", empty_years.index.tolist())


### 5.2 Compare each year with the previous year — V02

A year can be represented but have a sharply different record count. Use a change of at least 25% in either direction as a classroom review threshold. It does not prove that a count is wrong.

**Inspection question:** Which annual record counts change by at least 25% from the preceding year?

**Documentation:** [Series.shift()](https://pandas.pydata.org/docs/reference/api/pandas.Series.shift.html) · [Series.replace()](https://pandas.pydata.org/docs/reference/api/pandas.Series.replace.html) · [Series.abs()](https://pandas.pydata.org/docs/reference/api/pandas.Series.abs.html) · [DataFrame()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html).

**Annual-change screen:** From 100 records to 130 is +30%, and from 100 to 70 is -30%; both meet
the **at least 25% in either direction** review choice. From 100 to 125 meets it exactly; from
100 to 124 does not. A previous count of zero makes this percentage undefined rather than 0%.
The flag asks us to investigate activity and reporting, not to replace the count.


In [ ]:
# An annual change needs counts in chronological order and a meaningful previous-year
# denominator.

# For counts [100, 130, 90], shifting by one produces [missing, 100, 130] under the SAME year
# labels. Each row can now compare its own count with the preceding year.
previous_counts = year_counts.shift(1)

# A previous zero makes percentage change undefined. Replace it in a temporary denominator
# with missing, rather than inventing an increase of 0% or dividing by zero.
safe_previous_counts = previous_counts.replace(0, float("nan"))

# For 100 then 130 records, (130 - 100) / 100 * 100 gives +30%. For 100 then 70 it gives
# -30%. Label alignment keeps each current, previous and denominator value attached to the
# same comparison year.
year_changes = (year_counts - previous_counts) / safe_previous_counts * 100

# Absolute value treats a fall of 30% like a rise of 30%; both meet the 25% classroom REVIEW
# threshold. The threshold asks for context, not a declaration that either count is wrong.
large_change = year_changes.abs() >= 25

# Combine the year-labelled Series into one row per annual comparison. The first year and a
# year after a zero count have unavailable change values; a False review flag there does not
# mean they passed.
year_report = pd.DataFrame()
year_report["Records"] = year_counts
year_report["Previous records"] = previous_counts
year_report["Change percent"] = year_changes.round(2)
year_report["Review flag"] = large_change
print("Calculable annual changes:", year_changes.notna().sum())
# Inspect flagged years with their current and previous counts so actual activity, reporting
# changes and extract coverage can be considered before drawing a conclusion.
display(year_report.loc[large_change])


### 5.3 Examine combinations, not just separate lists — V01

A jurisdiction can occur somewhere and a year can occur somewhere while their combination is absent. A cross-tabulation reveals this gap.

**Inspection question:** Which expected jurisdiction–year combinations contain no eligible records?

**Documentation:** [pd.crosstab()](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html) · [DataFrame.reindex()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reindex.html) · [DataFrame.T](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.T.html) · [DataFrame.sum()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sum.html) · [DataFrame.any()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.any.html).


In [ ]:
# Seeing every year and every jurisdiction separately does not prove every jurisdiction-year
# pair has records.

# Group each eligible record using BOTH its jurisdiction and year. A record from CA in 2000
# contributes one to the CA row and 2000 column.
state_year_counts = pd.crosstab(coverage_data["state"], coverage_data["fire_year"])

# Include the full expected grid so absent categories or pairs still appear as zeros.
# Otherwise the very gaps we want to inspect could be omitted.
state_year_counts = state_year_counts.reindex(
    index=allowed_states, columns=expected_years, fill_value=0
)

# Transpose this small view only to put years down the page for CT and PR. The complete table
# and its count layout remain available for the following checks.
display(state_year_counts.loc[["CT", "PR"]].T)

# A True now refers to a GROUP combination with zero eligible records, not an individual
# source record. The same rule requires a different count unit from the missing-name check.
empty_combinations = state_year_counts == 0

# The first sum counts True cells down columns; the second adds those column totals. This
# gives the number of empty pairs across the grid.
empty_count = empty_combinations.sum().sum()
print("Combinations checked:", state_year_counts.size)
print("Combinations with no eligible records:", empty_count)

# Work across each jurisdiction row to find how many expected years lack records. A zero pair
# deserves coverage investigation; it is not a confirmed missing fire.
empty_years_per_state = empty_combinations.sum(axis=1)
states_with_gaps = empty_combinations.any(axis=1)
display(empty_years_per_state.loc[states_with_gaps])


## 6. Check relationships between dates and years

Coverage checks asked where records occur. We now compare fields within the same record.
Individually readable values can still disagree with each other or violate a logical relationship.


### 6.1 Compare reported and discovery years — L01

Annual grouping can change if fire_year differs from the year in discovery_date. Both a valid reported year and a usable discovery date are needed.

**Inspection question:** Do the two year values agree in the records we can compare?

**Documentation:** [Series.dt.year](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.year.html) · [Series.notna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.notna.html).

**Relationship rule examples:** Reported year 1993 with discovery in 1993 agrees; reported year
1993 with discovery in 1994 is flagged for investigation. A missing discovery date makes the
comparison untested. The disagreement alone does not identify the correct source definition.


In [ ]:
# Two fields can be individually readable yet place the same record in different annual
# groups.

# Extract just the calendar year from each usable discovery date. Keep source labels so the
# extracted year belongs to the same record as fire_year.
discovery_year = discovery_dates.dt.year

# Require both a valid reported year and a usable discovery date. Without either, equality
# cannot settle whether the two years agree.
year_can_be_checked = valid_year & discovery_dates.notna()
# For fire_year=1993 and discovery_date in 1994, the year comparison is True. The AND below
# restricts that disagreement to records meeting both prerequisites.
different_year = year_numbers != discovery_year
year_mismatch = year_can_be_checked & different_year
print("Records checked:", year_can_be_checked.sum())
print("Year disagreements:", year_mismatch.sum())
# Keep untested records visible. A disagreement identifies a relationship to investigate;
# source definitions are needed to decide whether either field is wrong.
print("Records not checked:", (~year_can_be_checked).sum())


### 6.2 Check date order — P03

A duration should not be negative. Check the ordering only where both dates could be read; a missing date prevents the comparison.

**Inspection question:** Does containment precede discovery in any record with both usable dates?

**Documentation:** [Series.notna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.notna.html) · [DataFrame.loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html).

**Date-order rule examples:** Discovery 10 July with containment 12 July meets the order rule;
containment 9 July is flagged. Containment on the same timestamp is not earlier, so it does not
violate this specific non-negative-duration condition. Missing either usable endpoint is untested.


In [ ]:
# A discovery-to-containment duration needs two usable endpoints and a non-negative order.

# If containment is absent, there is no second endpoint to compare. Such a record stays
# untested for date order even if the discovery date is valid.
both_dates_present = discovery_dates.notna() & contained_dates.notna()
# Discovery on 10 July and containment on 9 July would produce a negative duration, so the
# strict earlier-than comparison flags it. Equal endpoints produce zero duration and do not
# violate this particular rule.
contained_too_early = both_dates_present & (contained_dates < discovery_dates)
print("Records with both usable dates:", both_dates_present.sum())
print("Containment before discovery:", contained_too_early.sum())
print("Records not checked:", (~both_dates_present).sum())
# Show the two original endpoint values together. Zero flagged comparisons describes only
# records with both usable dates, not every record in the source.
display(raw.loc[contained_too_early, ["discovery_date", "contained_date"]].head(10))


## 7. Check that size and size class agree — P04

The size and class may each pass their individual rules yet disagree as a pair. Build an expected class from
each valid size, then compare it with the reported class. These are the teaching extract's continuous intervals.

| Class | Size in acres |
| --- | --- |
| A | Greater than 0, up to and including 0.25 |
| B | Greater than 0.25, below 10 |
| C | 10 up to, but not including, 100 |
| D | 100 up to, but not including, 300 |
| E | 300 up to, but not including, 1,000 |
| F | 1,000 up to, but not including, 5,000 |
| G | 5,000 or more |

Notice the boundaries: 0.25 belongs to A, 10 belongs to C and 5,000 belongs to G.

**How individual rules can pass while the pair disagrees:** A size of 10 acres is positive and
finite, and reported class `B` is an allowed A–G label. Both individual checks accept their
values, but the relationship is flagged because 10 acres belongs to `C` under these boundaries.
A missing size or an undocumented reported `H` cannot enter this eligible-pair comparison.


### 7.1 Build the expected-class Series

The expected class is a comparison value derived from a valid size. It must keep the same source-row labels so it can be matched to the original class.

**Inspection question:** Which class should each valid size fall into?

**Documentation:** [Series()](https://pandas.pydata.org/docs/reference/api/pandas.Series.html) · [Series.loc](https://pandas.pydata.org/docs/reference/api/pandas.Series.loc.html).


In [ ]:
# Derive an expected class independently from the positive finite size, then keep it
# alongside the original class for comparison.

# valid_size already requires size > 0. Adding <= 0.25 includes exactly 0.25 in A, while 0.26
# is outside A.
size_in_class_a = valid_size & (size_numbers <= 0.25)
print("Sizes that belong in A:", size_in_class_a.sum())
display(raw.loc[size_in_class_a, ["fire_size", "fire_class"]].head(5))

# Start with an unavailable expectation at EVERY original row label. Invalid or absent sizes
# must not accidentally receive a default class and appear checked.
expected_class = pd.Series(pd.NA, index=raw.index, dtype="string")

# Write a class only where that size interval applies. Labels preserve the link between each
# derived expectation and its own source row.
expected_class.loc[size_in_class_a] = "A"
# B excludes both 0.25 (already A) and 10 (belongs to C). These precise comparisons avoid
# gaps or overlapping assignments at the boundaries.
size_in_class_b = valid_size & (size_numbers > 0.25) & (size_numbers < 10)
expected_class.loc[size_in_class_b] = "B"

# Each later interval includes its lower boundary and excludes its upper one: 10 -> C, 100 ->
# D, 300 -> E, 1000 -> F and 5000 -> G. All conditions must hold for each assignment.
expected_class.loc[valid_size & (size_numbers >= 10) & (size_numbers < 100)] = "C"
expected_class.loc[valid_size & (size_numbers >= 100) & (size_numbers < 300)] = "D"
expected_class.loc[valid_size & (size_numbers >= 300) & (size_numbers < 1000)] = "E"
expected_class.loc[valid_size & (size_numbers >= 1000) & (size_numbers < 5000)] = "F"
# G has no finite upper boundary in this definition, but valid_size still excludes infinity.
# The resulting Series is a comparison, not a replacement for raw.fire_class.
expected_class.loc[valid_size & (size_numbers >= 5000)] = "G"


### 7.2 Compare expected and reported classes

The expected-class Series is ready. Compare it only where the size is valid and the reported class belongs to A–G.

**Inspection question:** Do any eligible records report a class inconsistent with their size?

**Documentation:** [Series.isin()](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html) · [Series.fillna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.fillna.html) · [DataFrame.copy()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html).


In [ ]:
# Compare expected and reported classes only where both sides meet their prerequisites.

# A valid size supplies an expectation; an allowed A-G label supplies a meaningful reported
# class. Missing classes and an undocumented H need their own findings rather than this
# disagreement test.
class_can_be_checked = valid_size & raw["fire_class"].isin(allowed_classes)

# Size 10 with reported C agrees, while size 10 with reported B disagrees. Matching row
# labels ensures the reported class is compared with that record's derived expectation.
class_mismatch = class_can_be_checked & (raw["fire_class"] != expected_class)
# Unknown Boolean comparisons cannot support a mismatch claim. Set them outside this mask
# while retaining the separately reported untested population.
class_mismatch = class_mismatch.fillna(False)
print("Size/class records checked:", class_can_be_checked.sum())
print("Size/class disagreements:", class_mismatch.sum())
print("Records not checked:", (~class_can_be_checked).sum())

# Display size, reported class and expectation together so the reason for a flag can be
# reconstructed. Select expected values by the evidence labels, not their positions in a
# different table.
class_evidence_preview = raw.loc[class_mismatch, ["fire_size", "fire_class"]].head(10).copy()
class_evidence_preview["Expected class"] = expected_class.loc[class_evidence_preview.index]
display(class_evidence_preview)


## 8. Investigate unusual measurements — O01

The rules above test defined expectations. Distribution checks instead flag values that are unusual relative to
other measurements. Fire sizes can be strongly skewed: a valid large fire may be far above the typical size.

**Defined rule versus statistical screen:** A size of `-2` acres violates the positive-area
definition. A size of 12 acres with an **illustrative** upper IQR fence of 11 is merely unusual
relative to the checked distribution; it can still be a genuine wildfire measurement.


### 8.1 Calculate the middle spread

Use only positive finite measurements. The interquartile range (IQR) is Q3 minus Q1: the width of the middle 50% of sizes.

**Inspection question:** Where does the middle half of the eligible sizes lie?

**Documentation:** [Series.quantile()](https://pandas.pydata.org/docs/reference/api/pandas.Series.quantile.html).


In [ ]:
# To describe the typical spread, first restrict the measurements to those that meet the
# positive finite-size prerequisite.

# Keep the source labels while filtering numeric sizes, so later largest-value or outlier
# results can locate their original records.
sizes = size_numbers.loc[valid_size]

# Q1 is the 25th-percentile boundary and Q3 the 75th-percentile boundary. Their difference is
# the width of the middle half of the eligible size distribution, still measured in acres.
q1 = sizes.quantile(0.25)
q3 = sizes.quantile(0.75)
# For an illustrative Q1=1 and Q3=5 acres, IQR=4 acres. This is a spread derived from the
# distribution, not a source-defined maximum valid fire size.
iqr = q3 - q1
# An empty eligible Series has no usable quartile boundaries. Even if the later flag count is
# zero, that empty population cannot demonstrate that sizes passed an outlier screen.
print("Positive finite sizes checked:", len(sizes))
print("Q1 (acres):", q1)
print("Q3 (acres):", q3)
print("IQR (acres):", iqr)


### 8.2 Flag values beyond the usual fences

Extend 1.5 IQR below Q1 and above Q3 to obtain review boundaries. A value outside them deserves investigation; the rule is not a reason to delete it.

**Inspection question:** How many eligible sizes fall outside the IQR fences?

**Documentation:** [Series.sum()](https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html) · [len()](https://docs.python.org/3.14/library/functions.html#len) · [round()](https://docs.python.org/3.14/library/functions.html#round).

**IQR review rule with worked numbers:** If Q1 is 1 acre and Q3 is 5 acres, IQR is 4 acres.
The lower fence is `1 - 1.5 × 4 = -5` and the upper fence is `5 + 1.5 × 4 = 11` acres.
A valid 12-acre size is flagged; 11 acres is exactly on the fence and is not flagged by the
strict comparisons. A missing, infinite or non-positive size stays outside this eligible screen.
These example boundaries explain the logic; the code computes the actual file's boundaries.


In [ ]:
# Extend the middle spread into review fences, then flag eligible sizes outside either fence.

# With illustrative Q1=1, Q3=5 and IQR=4 acres, the fences are -5 and 11 acres. The lower
# fence can be negative even though all eligible sizes are positive.
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

# | means either side is enough: below the lower fence OR above the upper. & also requires
# valid_size. Keep the full source-indexed Series so the decisions still select the correct
# rows in raw.
unusual_size = valid_size & ((size_numbers < lower_fence) | (size_numbers > upper_fence))
print("Lower fence:", lower_fence)
print("Upper fence:", upper_fence)
print("Eligible sizes:", len(sizes))
print("Flagged sizes:", unusual_size.sum())
# Divide flagged sizes by eligible sizes because only those entered this screen. With none
# available, neither a percentage nor an outlier conclusion is supported.
# The strict < and > comparisons do not flag a value exactly on a fence. Beyond-fence values
# deserve source and analytical review; the screen does not prove they are wrong.
if len(sizes) > 0:
    print("Flagged percentage:", round(unusual_size.sum() / len(sizes) * 100, 2))
else:
    print("Untested: no eligible sizes, so no percentage or conclusion.")


### 8.3 Inspect the upper tail

If many sizes are flagged, investigate the distribution and some large records before deciding whether any values are implausible.

**Inspection question:** How extreme are the largest sizes, and what other information accompanies them?

**Documentation:** [Series.describe()](https://pandas.pydata.org/docs/reference/api/pandas.Series.describe.html) · [Series.nlargest()](https://pandas.pydata.org/docs/reference/api/pandas.Series.nlargest.html) · [DataFrame.loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html).


In [ ]:
# A large flagged population may reflect a naturally long upper tail. Examine that tail
# before proposing any treatment.

# The 95th and 99th percentiles show where the upper few percent begin, helping explain why a
# mean can differ greatly from a median in a skewed distribution.
display(sizes.describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]))

# The largest numeric values retain their original source labels. Use those labels to
# retrieve names, locations and reported classes for context instead of inspecting size
# numbers alone.
largest_rows = sizes.nlargest(10).index
# A plausible named large fire may be a valid extreme observation. Confirming an error
# requires more evidence than its position in the size ranking.
display(raw.loc[largest_rows, ["fire_name", "state", "fire_size", "fire_class", "fire_cause"]])


## 9. Use simple plots to understand the flags

The numerical checks suggest a long upper tail. Use three views to understand it: the complete positive range,
a close view of small fires, and a logarithmic view that includes the full range.


### 9.1 Begin with the full positive range

An ordinary histogram counts how many sizes fall into each interval. Very large sizes may compress most records into the leftmost bars.

**Inspection question:** Can an ordinary scale show both typical and very large fires clearly?

**Documentation:** [Series.plot.hist()](https://pandas.pydata.org/docs/reference/api/pandas.Series.plot.hist.html) · [plt.title()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.title.html) · [plt.xlabel()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.xlabel.html) · [plt.ylabel()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.ylabel.html) · [plt.tight_layout()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.tight_layout.html) · [plt.show()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.show.html).


In [ ]:
# A full-range histogram shows the distribution, including how extreme sizes can compress the
# common small sizes on an ordinary axis.

import matplotlib.pyplot as plt

# Only plot when eligible sizes exist. An empty picture should not be presented as a
# distribution that demonstrates acceptable measurements.
if len(sizes) > 0:
    # Forty intervals divide the plotted size range and bar height counts records in each
    # interval. All positive finite sizes are included; very large maxima can make most
    # observations appear crowded at the left.
    sizes.plot.hist(bins=40, figsize=(8, 4))
    plt.title("Fire sizes: full positive range")
    # Label the horizontal axis in acres and the vertical axis in records so the bars cannot
    # be mistaken for total burned area.
    plt.xlabel("Fire size (acres)")
    plt.ylabel("Records")
    # Fit the labels and display this figure before the next cell, allowing each view to be
    # interpreted on its own.
    plt.tight_layout()
    plt.show()
else:
    print("No positive finite sizes to plot.")


### 9.2 Look more closely at small sizes

Zoom into sizes up to 10 acres to see the common small-fire pattern. This view deliberately excludes larger fires from the plot, not from the source data.

**Inspection question:** Where is the IQR upper fence relative to the small-fire distribution?

**Documentation:** [Series.plot.hist()](https://pandas.pydata.org/docs/reference/api/pandas.Series.plot.hist.html) · [plt.axvline()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.axvline.html) · [plt.legend()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.legend.html).


In [ ]:
# Use a closer view to inspect common small sizes and relate them to the IQR boundary; be
# clear that this picture uses a subset.

# Select sizes at or below 10 acres for this plot only. Large measurements remain in raw and
# in the full eligible Series, so this visual zoom is not a cleaning decision.
small_sizes = sizes.loc[sizes <= 10]
if len(small_sizes) > 0:
    small_sizes.plot.hist(bins=40, figsize=(8, 4))

    # The reference line locates the computed upper fence on the same acre scale. It helps
    # show whether many otherwise ordinary small fires can still lie beyond a
    # distribution-based screen.
    plt.axvline(upper_fence, color="red", linestyle="--", label="IQR upper fence")
    # State the subset in the title so the reader does not interpret this as the full
    # wildfire distribution.
    plt.title("Subset only: fire sizes up to 10 acres")
    plt.xlabel("Fire size (acres)")
    plt.ylabel("Records")
    # The legend explains the reference line rather than leaving its meaning to be guessed.
    # If the small-size subset is empty, skip this view and say why.
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No eligible sizes at or below 10 acres to plot.")


### 9.3 View the full range in orders of magnitude

A logarithm compresses a wide positive range while retaining every eligible size. Explain the transformed horizontal axis before interpreting the bars.

**Inspection question:** What does the full distribution look like when each step represents a tenfold size change?

**Documentation:** [np.log10()](https://numpy.org/doc/stable/reference/generated/numpy.log10.html) · [Series.plot.hist()](https://pandas.pydata.org/docs/reference/api/pandas.Series.plot.hist.html).


In [ ]:
# Compress the wide positive range using logarithms so both small and very large fires can be
# seen without discarding eligible measurements.

# Base-10 logs map 0.1, 1, 10 and 100 acres to -1, 0, 1 and 2. Adjacent steps now represent
# multiplication by ten, not the addition of one acre. Positive-size eligibility avoids
# logarithms of zero or negative sizes.
log_sizes = np.log10(sizes)
if len(log_sizes) > 0:
    # Every eligible measurement remains represented, but bins now divide logarithmic values.
    # This changes the visual scale, not the original measurements.
    log_sizes.plot.hist(bins=40, figsize=(8, 4))
    plt.title("Full positive range: logarithm of fire size")
    # Identify the transformed axis explicitly: a horizontal value of 2 means 100 acres, not
    # 2 acres. The vertical values still count records.
    plt.xlabel("log10(fire size in acres)")
    plt.ylabel("Records")
    plt.tight_layout()
    plt.show()
else:
    print("No positive finite sizes for a logarithmic view.")


## 10. Worked log entry: reported year and discovery year disagree — L01

Return to the within-record year check and turn it into a complete evidence trail. Our question is:
**Can we compare annual record counts using a consistent discovery year?** A disagreement can place the same record
in different annual groups. Investigate field definitions before deciding which source value, if either, should change.
The worked severity is **Low** for four disagreements in whole-file annual counts; a small affected subgroup could need a different judgement.

| Object we will build | What it contains | Meaning of one row |
| --- | --- | --- |
| `evidence_l01` | Original values supporting L01, plus reference labels | One affected source record |
| `log_entry` | A Python dictionary containing the answers for one issue | Not a table yet: field names paired with answers |
| `issue_log` | The dictionary converted into a DataFrame | One issue, however many records support it |

`source_row` preserves the original zero-based pandas row label from this unchanged import. It is not an incident ID
or a physical CSV line number. `issue_id` connects the evidence to its issue-log entry.

Some log fields are measured by code; others require your judgement. State the question before choosing severity.

| Severity | Meaning for the stated analysis |
| --- | --- |
| Low | Limited effect on the stated question. |
| Medium | A meaningful part of the comparison or grouping may be affected. |
| High | A central result cannot be supported reliably without a decision. |
| Not assigned | More investigation is needed before judging the effect. |

A percentage alone does not select a severity. A **proposed action** is a plan; **status** records progress;
**verification** explains how the later decision or treatment will be checked.


### 10.1 Identify and count records whose years disagree

Reuse year_numbers and discovery_dates from section 2, and valid_year from section 4.1. This complete rule keeps eligibility separate from disagreement.

**Inspection question:** Among records with both prerequisites, how many years disagree?

**Documentation:** [Series.dt.year](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.year.html) · [Series.notna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.notna.html) · [float()](https://docs.python.org/3.14/library/functions.html#float).

**Complete year-comparison rule:** First require a valid reported year and usable discovery
date; then flag inequality. Reported 1993/discovered 1994 meets the prerequisites and is flagged.
Reported 1993/missing discovery date does not meet the prerequisites and is untested. If four of
100 eligible comparisons disagree, the result is 4%, regardless of how many other records could
not be compared.


In [ ]:
# Rebuild the L01 finding as a complete sequence: derive a comparison year, restrict to
# testable records, and then measure disagreements.

# The extracted year keeps each discovery date's source label. That allows 1993 in fire_year
# to be compared with 1994 from the SAME record's date.
discovery_year = discovery_dates.dt.year
# A valid reported year AND a usable discovery date are both required. Missing prerequisites
# should not be hidden in a zero disagreement count.
year_can_be_checked = valid_year & discovery_dates.notna()

different_year = year_numbers != discovery_year
# The different-year condition alone is insufficient because unavailable values can produce
# misleading comparison results. Combine it with eligibility before counting.
year_mismatch = year_can_be_checked & different_year
year_mismatch_count = year_mismatch.sum()
# Count the full compared population, not all records regardless of their prerequisites. This
# is the denominator that tells us how frequent disagreements were among actual comparisons.
years_checked = year_can_be_checked.sum()

# Leave the percentage unavailable when no comparison could be made. A guarded division
# prevents both a runtime error and a false impression of a 0% failure rate.
mismatch_percent = float("nan")
if years_checked > 0:
    mismatch_percent = year_mismatch_count / years_checked * 100
else:
    print("Untested: no records meet both year-comparison prerequisites.")
print("Year disagreements:", year_mismatch_count)
print("Records compared:", years_checked)

# Four decimal places keep the tiny reference proportion visible; the count of four is more
# informative than a rounded 0.00%. Review the worked wording if your actual file gives
# another result.
print(f"Percentage affected: {mismatch_percent:.4f}%")


### 10.2 Build evidence with both year values

A reviewer needs the original reported year and discovery date, the year extracted from that date, and context for finding the record. Keep every disagreement as evidence.

**Inspection question:** Can someone see both values and trace the disagreement back to its source row?

**Documentation:** [DataFrame.loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [DataFrame.copy()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html) · [Series.loc](https://pandas.pydata.org/docs/reference/api/pandas.Series.loc.html) · [DataFrame.head()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html).


In [ ]:
# The evidence needs both source fields and the derived comparison year, so a reviewer can
# see why each L01 record was flagged.

# Keep original year, original discovery date and some context. A table of the disagreement
# flags alone would not reveal which two values differ.
evidence_columns = ["fire_name", "state", "fire_year", "discovery_date", "contained_date"]

# Select ALL disagreements and copy before annotating. The full table remains separate from
# the unmodified source.
evidence_l01 = raw.loc[year_mismatch, evidence_columns].copy()
# Preserve the original label before any index-free export. L01 links each evidence row to
# the single relationship finding.
evidence_l01["source_row"] = evidence_l01.index
evidence_l01["issue_id"] = "L01"

# Fetch derived years at the evidence source labels so each disagreement has the correct
# expected comparison beside it.
evidence_l01["discovery_year"] = discovery_year.loc[evidence_l01.index]

evidence_l01 = evidence_l01[[
    "issue_id", "source_row", "fire_name", "state", "fire_year",
    "discovery_date", "discovery_year", "contained_date"
]]
pd.set_option("display.max_columns", None)
print("Rows in full L01 evidence:", len(evidence_l01))
# The reference has four supporting records, so a ten-row display shows all of them. Another
# file may have more; the complete evidence must remain available even when the display is
# limited.
display(evidence_l01.head(10))


### 10.3 Record the measured facts

The reference extract has four disagreements among 1,880,465 eligible records. They support one issue-log entry, not four different issues. The percentage uses only records meeting both prerequisites.

**Inspection question:** What did we find, and how many records did we actually check?

**Documentation:** [dict](https://docs.python.org/3.14/library/stdtypes.html#dict) · [int()](https://docs.python.org/3.14/library/functions.html#int) · [round()](https://docs.python.org/3.14/library/functions.html#round).


In [ ]:
# Separate measured facts from written judgement so the count can be checked independently of
# our interpretation.

# A dictionary keeps named answers for ONE issue. Converting it to a table later gives one
# summary row even when thousands of source rows support that finding.
log_entry = {

    # Use L01 consistently in the log and evidence so the summary points to the correct
    # supporting records.
    "issue_id": "L01",

    # Describe what the check establishes. An absence or a disagreement is observable;
    # calling a source value an error would require additional evidence.
    "finding_type": "Year disagreement to investigate",

    "field": "fire_year / discovery_date",

    # year_mismatch_count counts the full flagged population. int stores a plain whole-number
    # count; the length of a display preview is not an alternative count.
    "affected_rows": int(year_mismatch_count),

    # years_checked is the population that could actually be tested. Excluding untestable
    # records from this denominator prevents us from making the issue appear smaller.
    "checked_rows": int(years_checked),

    # Keep four decimal places because rounding a very small non-zero result to two could
    # display 0.00%. An unavailable percentage remains unavailable; it is not evidence of a
    # pass.
    "percent": round(mismatch_percent, 4)
}


### 10.4 Connect the facts to the analysis and evidence

The counts describe the observation. We now explain why we checked it and where a reviewer can find the supporting records.

**Inspection question:** Could another student reproduce this finding and locate its evidence?

**Documentation:** [dict](https://docs.python.org/3.14/library/stdtypes.html#dict).


In [ ]:
# Add the reason for the check and a route back to its evidence; counts alone do not explain
# why the finding matters.

# The question determines relevance. Missing dates matter to duration; a date-year
# disagreement matters to annual grouping. A different question may lead to a different
# priority.
log_entry["question"] = 'Can we compare annual record counts using a consistent discovery year?'

# State the condition that would make the question answerable. This connects an acceptance or
# comparison rule to its purpose rather than treating it as an arbitrary instruction.
log_entry["expectation"] = (
    'For discovery-based annual groups, check whether the reported year agrees with '
    'the discovery year. Confirm what the source means by fire_year.'
)

# A notebook section and a reproducible condition let another person rerun the check. The
# issue label by itself would not explain how we arrived at the count.
log_entry["check_reference"] = (
    'Notebook 03, section 10.1: year_can_be_checked & (year_numbers != discovery_year).'
)

# This answer stores the name evidence_l01 as text; the actual DataFrame remains a separate
# object. The name tells the reviewer which table to consult, but does not save that table.
log_entry["evidence_table"] = "evidence_l01"

# Say whether every affected row was retained. Without that distinction, a ten-row example
# could be mistaken for the full evidence population.
log_entry["evidence_scope"] = (
    'All affected records are included; head(10) previews up to ten. It shows all four '
    'for the reference extract.'
)


### 10.5 Explain the effect and propose the next step

We have evidence, but inspection alone does not settle how the data should be treated. The following wording is a worked judgement for the stated question.

**Inspection question:** What could this finding change in our analysis, and what should happen next?


In [ ]:
# Write a justified decision proposal: the detection code supplies observations, while these
# answers explain their analytical consequences.

# Explain the consequence: reported year 1993 and discovery year 1994 place the same record
# in different annual groups.
log_entry["impact"] = (
    'Affected records can fall in different annual groups depending on which year '
    'field we use.'
)

# Priority follows the stated question, not a percentage cut-off. The explanation below must
# make the chosen priority defensible for this particular use.
log_entry["severity"] = 'Low'

# This is worked wording for the supplied reference finding. Compare it with your computed
# result and analysis population before reusing it.
log_entry["severity_reason"] = (
    'Only four records are involved in annual counts for the whole dataset. A small '
    'affected subgroup may need a different judgement.'
)

# Keep the decision as text at the inspection stage. Investigating recovery, defining a
# subset or agreeing a comparison policy is a proposal; assigning this answer changes no
# source values.
log_entry["proposed_action"] = (
    'Check the source definitions and these four records before choosing a year field '
    'or correcting any value.'
)

# Evidence exists, but the treatment has not been agreed or verified. Keeping the issue open
# prevents a proposed action being read as a completed correction.
log_entry["status"] = "Open - decision required"

# Name the later checks that could show the decision worked. Repeating the relevant check and
# reconciling counts is more convincing than merely writing "fixed".
log_entry["verification"] = (
    'Repeat the comparison after the decision. Record any confirmed correction or '
    'accepted difference, and check that annual counts follow the chosen year '
    'definition.'
)


### 10.6 Display, verify and optionally save the entry

The completed dictionary will become one row in `issue_log`. Keep that summary with the full `evidence_l01` evidence table. Each notebook creates its own one-entry log; they are not combined automatically. To add it to your shared log, match fields by meaning and update an existing L01 entry instead of duplicating it.

**Inspection question:** Does the recorded affected count agree with the evidence we kept?

**Documentation:** [DataFrame()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) · [pd.set_option()](https://pandas.pydata.org/docs/reference/api/pandas.set_option.html) · [DataFrame.T](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.T.html) · [DataFrame.to_csv()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html) · [len()](https://docs.python.org/3.14/library/functions.html#len).


In [ ]:
# Show the one-issue summary and compare its affected count with the complete supporting
# evidence before exporting.

# The outer list contains one dictionary, so Pandas interprets its keys as headings and its
# answers as one row. Re-running recreates this table; it does not append a second issue.
issue_log = pd.DataFrame([log_entry])

# Long written answers need to stay visible for review. Transposing the displayed table puts
# field names down the page; the stored issue_log still has one issue row.
pd.set_option("display.max_colwidth", None)
display(issue_log.T)

# These two counts should agree because one evidence row represents one flagged source
# record. A difference would suggest the evidence filter or the logged count came from a
# different check.
print("Rows in full evidence:", len(evidence_l01))
print("Affected rows recorded in log:", log_entry["affected_rows"])

# OPTIONAL: uncomment the next two commands to save both complete tables. index=False avoids
# a redundant index column; source_row still preserves traceability. Saving again replaces
# these CSVs, so retain reusable wording in the notebook.
# issue_log.to_csv("Issue_Log_L01.csv", index=False)
# evidence_l01.to_csv("evidence_l01.csv", index=False)


### Reference evidence for L01

These values are the worked findings retained from the supplied notebook. Compare them with your own run.

| Fire name | Reported fire_year | Year in discovery_date |
| --- | ---: | ---: |
| BONE | 1993 | 1994 |
| LITTLE POND | 1993 | 1994 |
| TRASK GRASS | 1993 | 1994 |
| INC. 253 | 2010 | 2009 |

For BONE, grouping by reported year uses 1993; grouping by discovery year uses 1994.
The evidence shows a disagreement, but does not decide which source definition should govern the analysis.


### Compare the three completed examples

Each notebook creates a separate one-entry log. Transfer those entries into your shared log;
running the notebooks does not combine them automatically.

| Issue | Evidence rows in the supplied file | Records checked for the percentage | Log rows needed |
| --- | ---: | --- | ---: |
| M02: missing containment date | 891,531 | Every record: 1,880,465 | 1 |
| T01: surrounding spaces in names | 77,704 | Present names: 919,986 | 1 |
| L01: two years disagree | 4 | Records with both prerequisites: 1,880,465 | 1 |

The shared log will contain **three entries** for these worked examples. The evidence files can contain
many more rows because they show the records supporting each entry. Do not add the three affected counts
to claim a total number of distinct problem records: the same record may appear in more than one check.

The repeated process is: choose a question, find affected records, keep useful evidence, record the facts,
explain the effect, propose a next step and keep the log with its evidence.

### Practise the same process: unusually large or small sizes — O01

An IQR flag means a measurement deserves a closer look. It does not prove that the measurement is wrong.
Keep the L01 entry before creating a new `log_entry` dictionary.

1. Choose a question, such as “How do unusually large fires affect average fire size?”
2. Reuse `unusual_size` from section 8.2 to select the records outside the IQR fences.
3. Create `evidence_o01` with `fire_name`, `state`, `fire_size` and `fire_class`, then add `source_row` and `issue_id` set to `O01`.
   Keep the lower and upper fence values with your check so another person can repeat it.
4. Use the full flag count, **284,405**, and the number of positive finite sizes checked, **1,880,465** for this file.
5. Write “284,405 records were flagged for review”, rather than claiming they are all incorrect.
6. Write one log entry pointing to `evidence_o01`. Explain a possible effect, such as unusually large values
   pulling the mean upwards. If you need more evidence to judge the importance, use **Not assigned** for severity.
7. Propose reviewing the source and context of flagged records. Keep the status Open and describe how
   confirmed errors and valid large fires would be distinguished before cleaning.

A plausible very large fire should not be deleted just because its size is above the IQR fence.


## 11. Bring the inspection together

The following is the worked conclusion for this supplied extract. Use your computed outputs to
replace the figures when inspecting another file. Counts refer to different populations and overlap.

| Finding | Evidence | What it means for later work |
| --- | --- | --- |
| S01: schema | Eight expected columns; no missing or extra fields | Structure is suitable for these checks; values still need inspection. |
| M01: names absent | 960,479 records | Name-based questions have incomplete coverage. |
| M02: containment absent | 891,531 records | Duration analysis needs a justified subset or recovered information. |
| C01: undefined cause | 166,723 records | Keep undefined cause visible in analysis and reporting. |
| R01: exact repetition | 365,548 members; 247,532 later repetitions | Resolve source identity before deleting any record. |
| R02: extra matches after text comparison | 43 newly flagged members; 22 extra later repetitions | Investigate candidates, keeping the original values. |
| T01: surrounding name spaces | 77,704 present names | Agree a name-comparison policy before cleaning. |
| T02: possible placeholders | 3,328 name records | Check their meaning rather than automatically replacing them. |
| T03: non-ASCII names | 22 records | Review encoding while preserving legitimate characters. |
| V01: coverage | No empty years; 34 empty jurisdiction–year combinations out of 1,248 | Establish reporting coverage before claiming missing or zero events. |
| V02: annual count changes | Three of 23 calculable changes meet the 25% threshold | Seek context; count changes are not inherently errors. |
| L01: year relationship | Four disagreements | Verify definitions and choose a consistent year field. |
| O01: IQR screen | 284,405 size flags | Investigate context; keep plausible large fires. |

Also retain useful passes: no source-header problems, no entirely missing or single-value columns,
no entirely missing rows, no whitespace-only names, no ASCII control-character flags, no numeric
conversion or infinity failures, no date-format or date-conversion failures among present dates,
no invalid category labels or numeric ranges, no size/class disagreements, and no reversed date
order **among the 988,934 records with both dates**.

### Choose checks that match the dataset

| Check family | When it is useful |
| --- | --- |
| Structure, types and missingness | Every tabular dataset; define expected fields first. |
| Repeated rows and keys | Review repetition; enforce uniqueness only where row meaning requires it. |
| Categories and text | Fields that represent labels, codes or text. |
| Conversion, ranges and relationships | Fields with documented numeric, date or domain expectations. |
| Period and group coverage | When the expected time span and group scope can be stated independently. |
| Distribution checks and plots | Meaningful measurements, with a clearly stated eligible population. |

When adapting these checks, change the column names, expected categories, date formats, units and
range rules. Record **not applicable** when a check has no relevant fields and **untested** when its
prerequisites or evidence are unavailable. Neither means “passed”. More specialised checks, such as
links between separate tables or expected sampling intervals, depend on the dataset and question.

### Write a justified conclusion

For preliminary exploration of **reported wildfire records**, this extract is usable with the documented
limitations. Duration results need a defined population with usable dates. Cause results should retain
the undefined category. Claims about unique fires require source identifiers. Comparisons across years
and jurisdictions need attention to coverage. Name-based grouping requires an agreed text policy.
The statistical size flags alone provide no reason to delete large fires.

Complete this sentence for your chosen question:

> For **[question]**, this dataset is **[usable with limitations / requires decisions / insufficient]**
> because **[inspection evidence]**. Before proceeding, we need to **[specific verification or decision]**.

Before moving to cleaning, check that every important issue has a reproducible check, a count and
denominator, evidence, an analytical effect, a justified severity (or “Not assigned”), a proposed action,
and a status. Keep the original data unchanged and distinguish an approved treatment from a proposed one.

**Examples when adapting the rules:** An event-duration rule is **not applicable** to a dataset
that has no event-date fields by design. It is **untested** for a wildfire record whose required
containment endpoint is unavailable. A whole-year rule flags `1998.5`; a size–class rule flags
10 acres reported as `B`. Use each field's own documentation rather than importing these
wildfire expectations into an unrelated dataset.


### Study check

Explain why a value can convert successfully but fail a range rule; why two individually valid fields can disagree;
why a record without both dates is untested for date order; and why an IQR flag does not justify automatic deletion.

For your final conclusion, connect each important finding to the analysis question it affects. Keep confirmed observations,
proposed treatments and verified corrections distinct. The inspection is complete when a reader can reproduce the findings
and understand which decisions are needed before cleaning or analysis.

**Use concrete cases in your explanation:** `2020` converts to a number but falls outside this
extract's period; 10 acres and class `B` each meet their individual rules but disagree as a pair;
a missing containment date prevents the date-order comparison; a valid extreme fire can be
flagged by the IQR screen. For each case, explain the prerequisite, condition and analytical consequence.
